In [ ]:
"""
=============================================================================
PROJETO: Classificador de Phishing e Engenharia Social (Fase 1)
=============================================================================

* INTENÇÃO DA MUDANÇA (CONTEXTO BRASILEIRO):
O modelo base (treinado apenas com dados em inglês traduzidos e robôs de
investimento) apresentou uma "cegueira cultural". O objetivo deste notebook
foi injetar contexto brasileiro moderno para que a IA aprendesse a identificar
fraudes locais, como golpes do Pix, falsas multas do Detran, falsas taxas dos
Correios e avisos falsos do gov.br.

* DADOS UTILIZADOS:
- Dataset original (WhatsApp/Telegram)[cite: 1].
- Amostra traduzida do dataset SMS Spam Collection[cite: 1].
- Injeção de 959 mensagens sintéticas geradas por IA focadas puramente no
  cenário brasileiro ('phishing_sintetico_br.csv')[cite: 1].
- Dataset consolidado salvo como 'meu_super_dataset_phishing_v2.csv', gerando
  um treinamento balanceado com 11.054 mensagens[cite: 1].

* RESULTADOS PRELIMINARES:
O modelo obteve um desempenho excelente no teste inicial de validação (94%
de acurácia)[cite: 1]. No entanto, testes diagnósticos revelaram um fenômeno
de "Shortcut Learning" (Aprendizado por Atalho). Ao remover a tag '[URL]'
dos testes, a acurácia global do modelo caiu drasticamente para 82%[cite: 1].
A IA estava usando a existência de um link como muleta estrutural para
classificar as mensagens, ignorando o contexto semântico do texto.

* POSSÍVEIS MELHORIAS (PRÓXIMO PASSO):
Aplicar a técnica de "Text Dropout" (Ocultação Aleatória) na base de dados
para remover a tag '[URL]' de forma parcial. Isso forçará o modelo a ler
a semântica e a intenção da frase, preparando-o para golpes de engenharia
social de múltiplas etapas, onde o cibercriminoso não envia o link de imediato.
=============================================================================
"""

In [ ]:
# 1. Instalar a NOVA biblioteca oficial do Google
!pip install -q google-genai pandas
!pip install -q datasets deep-translator pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.9 MB/s eta 0:00:00


In [ ]:
# 1. Instalar as bibliotecas necessárias
import json
import time
from google import genai
import pandas as pd
from datasets import load_dataset
from deep_translator import GoogleTranslator
from tqdm import tqdm
import pandas as pd
from datasets import load_dataset, Dataset
from deep_translator import GoogleTranslator
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import torch
#!git clone -b desenv https://github.com/jmmfilho/sec-fraudimabr.git

In [ ]:
# 2. Configure sua chave de API
API_KEY = ""
client = genai.Client(api_key=API_KEY)
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-p

In [ ]:



# Usaremos o modelo atualizado e recomendado
# Atualizado para a versão ativa mais recente do Flash
MODEL_ID = 'gemini-3.5-flash-lite'

# 3. Definir as categorias e a meta
META_POR_CATEGORIA = 160
MENSAGENS_POR_LOTE = 20

categorias_fraude = [
    "Logística: Correios (taxa alfandegária, pacote retido), Mercado Livre, Shopee.",
    "Governo/Impostos: gov.br (atualização), Receita Federal (malha fina), Detran (CNH, multa NIC).",
    "Falso Familiar/Conhecido: 'Oi pai mudei de número', 'Mãe meu celular quebrou', Pix urgente.",
    "Falsos Empregos: Avaliador de roupas (Shein), curtidor de vídeos (TikTok), Amazon.",
    "Serviços e Dívidas: Serasa (feirão limpa nome), contas de luz cortadas, Netflix recusado.",
    "Bancos/Cartões: Bloqueio de chave Pix, compras aprovadas (Itaú, Nubank, Bradesco)."
]

dados_sinteticos = []

# 4. Função para gerar o prompt dinâmico
def criar_prompt(categoria, quantidade):
    return f"""
    Aja como um especialista em Red Team. Gere exatamente {quantidade} mensagens curtas (10 a 40 palavras) de phishing por SMS/WhatsApp para o cenário brasileiro.

    CENÁRIO OBRIGATÓRIO PARA ESTE LOTE:
    {categoria}

    REGRAS DE ESTILO:
    1. Seja caótico: Inclua abreviações (vc, tb, pq, p/), falta de acentuação e pontuação incorreta. Pareça um humano digitando com pressa ou um golpista amador.
    2. Use a tag [URL] no lugar de links reais.
    3. Crie urgência, curiosidade ou desespero.

    FORMATO DE SAÍDA:
    Retorne APENAS um array JSON válido contendo as strings das mensagens. Não adicione crases de formatação (```json), explicações ou títulos. Apenas o array: ["msg 1", "msg 2"]
    """

print("Iniciando a fábrica de dados sintéticos...\n")

# 5. O Loop de Geração
for categoria in categorias_fraude:
    nome_categoria = categoria.split(":")[0]
    print(f"Gerando categoria: {nome_categoria}...")

    mensagens_geradas_nesta_categoria = 0

    while mensagens_geradas_nesta_categoria < META_POR_CATEGORIA:
        try:
            prompt = criar_prompt(categoria, MENSAGENS_POR_LOTE)

            # Chamada usando a NOVA API
            resposta = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt
            )

            # Limpar formatação residual
            texto_limpo = resposta.text.replace("```json", "").replace("```", "").strip()

            # Converter a string JSON para uma lista do Python
            lote_mensagens = json.loads(texto_limpo)

            for msg in lote_mensagens:
                dados_sinteticos.append({"texto": msg, "label": 1}) # 1 = Fraude
                mensagens_geradas_nesta_categoria += 1

            print(f"  -> +{len(lote_mensagens)} mensagens ({mensagens_geradas_nesta_categoria}/{META_POR_CATEGORIA})")

            # Pausa de 4 segundos para respeitar o limite gratuito da API
            time.sleep(4)

        except Exception as e:
            print(f"  -> Erro na geração. Limite de cota ou erro de JSON. Aguardando para tentar novamente... Erro: {e}")
            time.sleep(6)

# 6. Salvar os resultados
df_sintetico = pd.DataFrame(dados_sinteticos)

# Remover possíveis duplicatas
df_sintetico = df_sintetico.drop_duplicates(subset=['texto'])

df_sintetico.to_csv("phishing_sintetico_br.csv", index=False)

print(f"\n✅ Concluído! Arquivo salvo como 'phishing_sintetico_br.csv'.")
print(f"Total de mensagens únicas geradas: {len(df_sintetico)}")

Iniciando a fábrica de dados sintéticos...

Gerando categoria: Logística...
  -> +20 mensagens (20/160)
  -> +20 mensagens (40/160)
  -> +20 mensagens (60/160)
  -> +20 mensagens (80/160)
  -> +20 mensagens (100/160)
  -> +20 mensagens (120/160)
  -> +20 mensagens (140/160)
  -> +20 mensagens (160/160)
Gerando categoria: Governo/Impostos...
  -> +20 mensagens (20/160)
  -> +20 mensagens (40/160)
  -> +20 mensagens (60/160)
  -> +20 mensagens (80/160)
  -> +20 mensagens (100/160)
  -> +20 mensagens (120/160)
  -> +20 mensagens (140/160)
  -> +20 mensagens (160/160)
Gerando categoria: Falso Familiar/Conhecido...
  -> +20 mensagens (20/160)
  -> +20 mensagens (40/160)
  -> +20 mensagens (60/160)
  -> +20 mensagens (80/160)
  -> +20 mensagens (100/160)
  -> +20 mensagens (120/160)
  -> +20 mensagens (140/160)
  -> +20 mensagens (160/160)
Gerando categoria: Falsos Empregos...
  -> +20 mensagens (20/160)
  -> +20 mensagens (40/160)
  -> +20 mensagens (60/160)
  -> +20 mensagens (80/160)
  ->

In [ ]:


# 2. Carregar o seu dataset original (WhatsApp/Telegram)
print("Carregando dataset original...")
df_original = pd.read_csv('sec-fraudimabr/data/dataframe_rotulado_completo.csv')
df_original = df_original[['text_content', 'Golpe']].dropna()
# Padronizando os nomes das colunas
df_original.rename(columns={'text_content': 'texto', 'Golpe': 'label'}, inplace=True)

# 3. Carregar o SMS Spam Collection (em inglês) direto da Hugging Face
print("Baixando SMS Spam Collection...")
dataset_sms = load_dataset("ucirvine/sms_spam", split="train")
df_sms_en = dataset_sms.to_pandas()

# No SMS Spam, 'label' 0 = ham (normal) e 1 = spam (golpe)
df_sms_spam = df_sms_en[df_sms_en['label'] == 1].sample(500, random_state=42, replace=True)
df_sms_ham = df_sms_en[df_sms_en['label'] == 0].sample(500, random_state=42)
df_sms_amostra = pd.concat([df_sms_spam, df_sms_ham])

# 4. Traduzir para o Português
print("Traduzindo 1000 mensagens para o português (isso levará cerca de 2 a 3 minutos)...")
tradutor = GoogleTranslator(source='en', target='pt')
textos_traduzidos = []

for texto in tqdm(df_sms_amostra['sms']):
    try:
        traducao = tradutor.translate(texto)
        textos_traduzidos.append(traducao)
    except:
        # Se a API falhar em uma mensagem específica, mantém a original
        textos_traduzidos.append(texto)

# Criar um DataFrame com os dados traduzidos
df_sms_pt = pd.DataFrame({
    'texto': textos_traduzidos,
    'label': df_sms_amostra['label'].values
})

# ============================================================
# NOVO PASSO: INJETAR OS DADOS SINTÉTICOS BRASILEIROS
# ============================================================
print("Carregando o dataset sintético gerado com o Gemini...")
try:
    df_sintetico = pd.read_csv("phishing_sintetico_br.csv")
    # Limpeza de segurança para garantir que não haja linhas vazias
    df_sintetico = df_sintetico[['texto', 'label']].dropna()
    print(f"-> {len(df_sintetico)} mensagens sintéticas carregadas com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os dados sintéticos: {e}")
    # Cria um dataframe vazio caso o arquivo falhe para não quebrar o código
    df_sintetico = pd.DataFrame(columns=['texto', 'label'])
# ============================================================

# 5. Mesclar os TRÊS datasets em um Super Dataset Final
print("Mesclando e construindo o dataset final...")
df_final = pd.concat([df_original, df_sms_pt, df_sintetico], ignore_index=True)

# Embaralhar os dados finais (importante para o treinamento)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# Salvar o novo dataset pronto para treino
df_final.to_csv("meu_super_dataset_phishing_v2.csv", index=False)

print(f"\n✅ Super Dataset V2 criado e salvo como 'meu_super_dataset_phishing_v2.csv'!")
print(f"Total de mensagens combinadas: {len(df_final)}")
print("\nDistribuição Final das Classes (0 = Normal, 1 = Golpe):")
print(df_final['label'].value_counts())

Carregando dataset original...
Baixando SMS Spam Collection...


README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  359kB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

Traduzindo 1000 mensagens para o português (isso levará cerca de 2 a 3 minutos)...


100%|██████████| 1000/1000 [08:08<00:00,  2.05it/s]


Carregando o dataset sintético gerado com o Gemini...
-> 959 mensagens sintéticas carregadas com sucesso!
Mesclando e construindo o dataset final...

✅ Super Dataset V2 criado e salvo como 'meu_super_dataset_phishing_v2.csv'!
Total de mensagens combinadas: 17061

Distribuição Final das Classes (0 = Normal, 1 = Golpe):
label
1    11534
0     5527
Name: count, dtype: int64


In [ ]:


print("Iniciando a preparação dos dados...")

# ==========================================
# PASSO 1: PREPARAÇÃO E BALANCEAMENTO
# ==========================================

# 1. Carregar o novo Super Dataset
df = pd.read_csv("meu_super_dataset_phishing_v2.csv")

# 2. Balanceamento Estrito (Undersampling)
# Descobre qual é a classe com menos exemplos e iguala a outra para ter o mesmo tamanho
tamanho_menor_classe = df['label'].value_counts().min()

df_normal = df[df['label'] == 0].sample(tamanho_menor_classe, random_state=42)
df_golpe = df[df['label'] == 1].sample(tamanho_menor_classe, random_state=42)

# Junta tudo e embaralha
df_balanceado = pd.concat([df_normal, df_golpe]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset balanceado com sucesso! Total: {len(df_balanceado)} mensagens.")
print(df_balanceado['label'].value_counts())

# 3. Divisão de Treino (80%) e Validação (20%)
df_train, df_val = train_test_split(df_balanceado, test_size=0.3, random_state=42)

# As colunas já estão como 'texto' e 'label', vamos renomear para o padrão Hugging Face
df_train.columns = ['text', 'labels']
df_val.columns = ['text', 'labels']

# Converter Pandas para Dataset da Hugging Face
train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val, preserve_index=False)

# ==========================================
# PASSO 2: TREINAMENTO (API MODEL)
# ==========================================

# 1. Inicializar Tokenizer e Modelo
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Função de Tokenização
def tokenizar_funcao(exemplos):
    return tokenizer(exemplos["text"], padding=False, truncation=True, max_length=128)

print("Tokenizando os textos...")
tokenized_train = train_dataset.map(tokenizar_funcao, batched=True)
tokenized_val = val_dataset.map(tokenizar_funcao, batched=True)

# 3. Configurar Data Collator (Otimiza a memória da GPU preenchendo os textos dinamicamente)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Configurar Hiperparâmetros
training_args = TrainingArguments(
    output_dir="./resultados_bert_phishing",
    eval_strategy="epoch",             # Avalia no fim de cada época
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4 ,               # 3 épocas é o padrão ouro para BERT
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# 5. Inicializar o Treinador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,        # Parâmetro atualizado (substitui o antigo 'tokenizer')
    data_collator=data_collator
)

# 6. INICIAR O TREINAMENTO
print("Iniciando o treinamento na GPU...")
trainer.train()

# 7. Salvar o modelo final
model.save_pretrained("./meu_modelo_api_phishing")
tokenizer.save_pretrained("./meu_modelo_api_phishing")
print("✅ Modelo salvo com sucesso na pasta './meu_modelo_api_phishing'!")

Iniciando a preparação dos dados...
Dataset balanceado com sucesso! Total: 11054 mensagens.
label
1    5527
0    5527
Name: count, dtype: int64


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (8 outputs left out)

Map:   0%|          | 0/3317 [00:00<?, ? examples/s]

Iniciando o treinamento na GPU...


Epoch,Training Loss,Validation Loss
1,No log,0.040544


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
# Carregar o pipeline de classificação com o modelo que você acabou de treinar
classificador = pipeline(
    "text-classification",
    model="./meu_modelo_api_phishing",
    tokenizer="./meu_modelo_api_phishing",
    device=0 if torch.cuda.is_available() else -1 # Usa GPU se disponível
)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
print("Carregando o modelo salvo...")
# 1. Carregar o modelo treinado diretamente da pasta onde foi salvo
classificador = pipeline(
    "text-classification",
    model="./meu_modelo_api_phishing",
    tokenizer="./meu_modelo_api_phishing",
    device=0 if torch.cuda.is_available() else -1, # Usa GPU para acelerar a validação
    truncation=True,    # Corta textos que forem maiores que o limite
    max_length=128
)

# 2. Extrair os textos e os rótulos (labels) do dataset de validação (df_val)
# Nota: Assumimos que a variável df_val ainda está na memória do Colab.
textos_val = df_val['text'].tolist()
y_true_val = df_val['labels'].tolist()

print(f"Realizando predições em {len(textos_val)} mensagens de validação. Isso pode levar alguns segundos...")
# 3. Fazer as predições em lote
resultados_val = classificador(textos_val)

# 4. Extrair os valores preditos (0 ou 1) da resposta do modelo
y_pred_val = []
for resultado in resultados_val:
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Extraímos apenas o número.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred_val.append(label_previsto)

# 5. Calcular as métricas
acuracia_val = accuracy_score(y_true_val, y_pred_val)
f1_val = f1_score(y_true_val, y_pred_val, average='binary')

# 6. Exibir o Relatório Completo
print("\n" + "="*50)
print(" RESULTADOS REAIS NO DATASET DE VALIDAÇÃO (20%)")
print("="*50)
print(f"Acurácia Global: {acuracia_val:.4f} ({acuracia_val * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1_val:.4f} ({f1_val * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true_val, y_pred_val, target_names=["Normal (0)", "Phishing (1)"]))

Carregando o modelo salvo...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Realizando predições em 2211 mensagens de validação. Isso pode levar alguns segundos...

 RESULTADOS REAIS NO DATASET DE VALIDAÇÃO (20%)
Acurácia Global: 0.9964 (99.64%)
F1-Score (Fraudes): 0.9963 (99.63%)

Relatório Detalhado:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00      1137
Phishing (1)       1.00      1.00      1.00      1074

    accuracy                           1.00      2211
   macro avg       1.00      1.00      1.00      2211
weighted avg       1.00      1.00      1.00      2211



In [ ]:
# 1. Modificando as mensagens para incluir o rótulo real (Ground Truth)
# Formato: ("Texto da mensagem", rótulo_real)
# Assumindo que 0 = Normal e 1 = Phishing (verifique a ordem do seu CSV)
mensagens_teste = [
    # --- MENSAGENS NORMAIS (Label 0) ---
    ("Oi, não esquece de passar na padaria voltando do trabalho.", 0),
    ("A reunião foi transferida para a sala 3, avisem os demais.", 0),
    ("Você viu o jogo ontem? Que golaço no final!", 0),
    ("Seu pedido Magalu 4920 foi entregue ao destinatário.", 0),
    ("Feliz Natal, família! Que a noite de vocês seja iluminada.", 0),
    ("Preciso que você me envie o PDF daquela apresentação até meio-dia.", 0),
    ("Estou no trânsito, devo atrasar uns 10 minutinhos.", 0),
    ("Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.", 0),
    ("Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.", 0),
    ("Bora marcar um futebol no fim de semana?", 0),
    ("Amor, a ração do cachorro acabou, compra mais depois.", 0),
    ("A sua fatura da Vivo com vencimento em 10/08 está disponível no app.", 0),
    ("Mãe, cheguei bem no hotel. Mais tarde te ligo.", 0),
    ("Alguém tem o contato de um bom encanador pra me passar?", 0),
    ("A professora de inglês mandou o link da aula no grupo.", 0),
    ("Oi, desculpa a demora pra responder, tava em reunião.", 0),
    ("Você vai almoçar em casa hoje ou no restaurante?", 0),
    ("Parabéns, amigo! Muitas felicidades e sucesso sempre.", 0),
    ("Uber: Seu motorista Carlos está chegando em 2 minutos.", 0),
    ("O boleto do condomínio já foi enviado pro seu email.", 0),
    ("Vou viajar no feriado, depois a gente se vê.", 0),
    ("Tem pão fresco saindo agora, vai querer que eu leve?", 0),
    ("A vacinação da gripe começou no posto aqui da esquina.", 0),
    ("Não consegui falar com você, me liga quando puder.", 0),
    ("Confirmando o nosso jantar de amanhã às 20h. Até lá!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("BANCO CENTRAL: Há um Pix de R$ 850 retido em seu nome. Acesse para liberar: ", 1),
    ("C6 Bank: Seu cartão foi bloqueado por segurança. Valide sua identidade imediatamente: ", 1),
    ("Mãe, meu celular quebrou, salva esse meu número novo provisório.", 1),
    ("Olá, somos do RH da Amazon. Temos vaga de meio período para ganhar de 200 a 500 por dia. Clique: ", 1),
    ("Detran informa: Suspensão de CNH agendada. Pague o débito para evitar o cancelamento: ", 1),
    ("Aviso gov.br: Sua conta será desativada em 2h. Atualize seus dados cadastrais em ", 1),
    ("CORREIOS: Seu pedido foi retido na alfândega. Taxa de R$ 22,50 pendente. Pague aqui: ", 1),
    ("Netflix: O pagamento do mês falhou. Atualize seu cartão de crédito para não perder o acesso: ", 1),
    ("Promoção O Boticário! Responda o questionário e ganhe um kit grátis de Dia das Mães: ", 1),
    ("Itaú: Identificamos um acesso no Rio de Janeiro. Caso não tenha sido você, conteste em ", 1),
    ("Você tem 20.000 pontos Livelo expirando hoje! Troque por milhas ou produtos em ", 1),
    ("Serasa Consumidor: Pague sua dívida com 95% de desconto no Feirão Limpa Nome. Pix oficial: ", 1),
    ("O Governo aprovou um Auxílio Gás de R$ 120. Consulte se seu CPF tem direito: ", 1),
    ("Oi, encontrei essas fotos suas da festa de ontem rolando na internet, dá uma olhada ", 1),
    ("Shopee: Parabéns, você foi selecionado para avaliar produtos. Renda diária de R$ 300. Chame no WhatsApp: ", 1),
    ("Nubank: Sua compra no valor de R$ 2.500 no MercadoPago foi aprovada. Para cancelar, clique: ", 1),
    ("Aviso Receita Federal: Sua declaração de imposto de renda caiu na malha fina. Veja a divergência: ", 1),
    ("WhatsApp: Insira o código de 6 dígitos que você recebeu por SMS para não perder sua conta.", 1),
    ("Caixa Tem: Seu saque extraordinário foi liberado. Acesse a conta para transferir o valor: ", 1),
    ("Santander: Seu aplicativo de segurança está desatualizado. Evite multas atualizando no link: ", 1),
    ("Oportunidade Única! Invista R$ 50 no nosso robô de Pix e receba R$ 500 em 10 minutos. Garanta sua vaga: ", 1),
    ("Claro: Sua fatura em aberto pode ser paga com 50% de desconto hoje no Pix. Gere a chave em: ", 1),
    ("Atenção: Ação judicial protocolada em seu CPF. Visualize o processo e mandado de intimação aqui: ", 1),
    ("Empréstimo liberado de R$ 10.000 para negativados sem consulta ao SPC! Assine o contrato online: ", 1),
    ("Mercado Livre: Seu acesso está suspenso temporariamente por atividade suspeita. Confirme sua titularidade", 1)
]

# 2. Separar os textos e os rótulos verdadeiros para processamento
textos = [item[0] for item in mensagens_teste]
y_true = [item[1] for item in mensagens_teste]

# 3. Fazer as predições usando o pipeline da Hugging Face
resultados = classificador(textos)

# 4. Extrair as predições do modelo (y_pred)
y_pred = []
print("--- DETALHAMENTO DAS MENSAGENS ---\n")
for texto, real, resultado in zip(textos, y_true, resultados):
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Pegamos apenas o número final e convertemos para inteiro.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred.append(label_previsto)

    print(f"Mensagem: {texto}")
    print(f"Real: {real} | Previsto: {label_previsto} | Certeza: {resultado['score']:.4f}\n")

# 5. Calcular e exibir as métricas
acuracia = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='binary') # 'binary' é ideal para problemas de duas classes

print("="*40)
print(" RESULTADOS GERAIS DA AVALIAÇÃO")
print("="*40)
print(f"Acurácia Global: {acuracia:.4f} ({acuracia * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1:.4f} ({f1 * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Phishing (1)"]))

--- DETALHAMENTO DAS MENSAGENS ---

Mensagem: Oi, não esquece de passar na padaria voltando do trabalho.
Real: 0 | Previsto: 0 | Certeza: 0.9998

Mensagem: A reunião foi transferida para a sala 3, avisem os demais.
Real: 0 | Previsto: 0 | Certeza: 0.9998

Mensagem: Você viu o jogo ontem? Que golaço no final!
Real: 0 | Previsto: 0 | Certeza: 0.9996

Mensagem: Seu pedido Magalu 4920 foi entregue ao destinatário.
Real: 0 | Previsto: 0 | Certeza: 0.9782

Mensagem: Feliz Natal, família! Que a noite de vocês seja iluminada.
Real: 0 | Previsto: 0 | Certeza: 0.9998

Mensagem: Preciso que você me envie o PDF daquela apresentação até meio-dia.
Real: 0 | Previsto: 0 | Certeza: 0.9998

Mensagem: Estou no trânsito, devo atrasar uns 10 minutinhos.
Real: 0 | Previsto: 0 | Certeza: 0.9997

Mensagem: Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.
Real: 0 | Previsto: 0 | Certeza: 0.9631

Mensagem: Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.
Real: 0 | Previs